# Notebook 02 — Cleaning & Feature Engineering
Drop leakage columns, handle nulls, parse dates, and engineer all model features.
Saves cleaned versions of both project-level and disaster-level datasets.


In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../')
from utils import get_season, add_prior_disasters, data_summary, PROJECT_BINS, DISASTER_BINS

PROCESSED = '../data/processed/'

proj  = pd.read_csv(PROCESSED + 'merged_project_level.csv',  low_memory=False)
disas = pd.read_csv(PROCESSED + 'merged_disaster_level.csv', low_memory=False)
print('Project-level:  ', proj.shape)
print('Disaster-level: ', disas.shape)

Project-level:   (810656, 33)
Disaster-level:  (1766, 14)


## 2.1 Drop Leakage Columns
`projectSize` is set by the eligible amount — a direct derivative of the label. Drop it.
`projectAmount` and `totalObligated` also leak the label.


In [2]:
LEAKAGE = ['projectSize', 'projectAmount', 'totalObligated', 'mitigationAmount',
           'lastObligationDate', 'firstObligationDate']

proj  = proj.drop(columns=[c for c in LEAKAGE if c in proj.columns])
print('Project-level after dropping leakage:', proj.shape)


Project-level after dropping leakage: (810656, 27)


## 2.2 Parse Dates


In [3]:
DATE_COLS = ['declarationDate', 'incidentBeginDate', 'incidentEndDate']

for col in DATE_COLS:
    if col in proj.columns:
        proj[col] = pd.to_datetime(proj[col], errors='coerce')
    if col in disas.columns:
        disas[col] = pd.to_datetime(disas[col], errors='coerce')

print('Date columns parsed.')


Date columns parsed.


## 2.3 Handle Nulls


In [4]:
print('PROJECT-LEVEL — null counts (top 15):')
nulls = proj.isnull().sum().sort_values(ascending=False)
print(nulls[nulls > 0].head(15))

# Impute numeric county features with median (counties without Census/NRI data)
for col in ['median_income', 'poverty_rate', 'population', 'risk_score']:
    if col in proj.columns:
        proj[col] = proj[col].fillna(proj[col].median())
    if col in disas.columns:
        disas[col] = disas[col].fillna(disas[col].median())

# Fill missing categorical values
proj['damageCategoryCode'] = proj['damageCategoryCode'].fillna('Unknown')

# Drop rows with no target or no date (can't use these at all)
proj  = proj.dropna(subset=['federalShareObligated', 'incidentBeginDate'])
disas = disas.dropna(subset=['total_federal_share', 'incidentBeginDate'])

print(f'\nAfter null handling -> Project: {proj.shape} | Disaster: {disas.shape}')


PROJECT-LEVEL — null counts (top 15):
risk_score       49701
risk_rating      26043
population       26043
poverty_rate     20890
median_income    20890
county           13620
applicantId          5
dtype: int64

After null handling -> Project: (810656, 27) | Disaster: (1766, 14)


## 2.3b Canonical Incident Type Mapping
FEMA uses inconsistent spellings across years ('Severe Storm(s)', 'Severe Storms', 'Severe Storm').
Map all variants to 7 canonical categories so the OHE encoder learns stable patterns.

In [ ]:
INCIDENT_TYPE_CANONICAL = {
    # Flood
    'Flood':                  'Flood',
    'Flooding':               'Flood',
    'Dam/Levee Break':        'Flood',
    # Hurricane / Tropical
    'Hurricane':              'Hurricane',
    'Typhoon':                'Hurricane',
    'Tropical Storm':         'Hurricane',
    'Coastal Storm':          'Hurricane',
    # Severe Storm
    'Severe Storm(s)':        'Severe Storm',
    'Severe Storms':          'Severe Storm',
    'Severe Storm':           'Severe Storm',
    'Thunderstorms':          'Severe Storm',
    'Thunderstorm Winds':     'Severe Storm',
    # Winter
    'Snow':                   'Winter Storm',
    'Snowstorm':              'Winter Storm',
    'Blizzard':               'Winter Storm',
    'Ice Storm':              'Winter Storm',
    'Severe Ice Storm':       'Winter Storm',
    'Freezing':               'Winter Storm',
    'Winter Storm':           'Winter Storm',
    'Winter Storms':          'Winter Storm',
    'Cold Wave':              'Winter Storm',
    # Tornado
    'Tornado':                'Tornado',
    'Tornadoes':              'Tornado',
    # Wildfire
    'Fire':                   'Wildfire',
    'Wildfire':               'Wildfire',
    'Forest Fires':           'Wildfire',
    # Earthquake
    'Earthquake':             'Earthquake',
    # Everything else → Other
    'Biological':             'Other',
    'Chemical':               'Other',
    'Toxic Substances':       'Other',
    'Terrorist':              'Other',
    'Human Cause':            'Other',
    'Drought':                'Other',
    'Mud/Landslide':          'Other',
    'Landslide':              'Other',
    'Volcanic Eruption':      'Other',
    'Tsunami':                'Other',
    'Other':                  'Other',
}

for df in [proj, disas]:
    before = df['incidentType'].nunique()
    df['incidentType'] = df['incidentType'].map(INCIDENT_TYPE_CANONICAL).fillna('Other')
    after  = df['incidentType'].nunique()
    print(f'incidentType: {before} variants → {after} canonical categories')

print('\nCanonical distribution (disasters):')
print(disas['incidentType'].value_counts())

## 2.4 Feature Engineering — Time Features


In [5]:
for df in [proj, disas]:
    df['declaration_lag_days']   = (df['declarationDate']   - df['incidentBeginDate']).dt.days
    df['incident_duration_days'] = (df['incidentEndDate']   - df['incidentBeginDate']).dt.days
    df['incident_season']        = df['incidentBeginDate'].dt.month.apply(get_season)
    df['incident_year']          = df['incidentBeginDate'].dt.year

# Clip negative lags (data entry errors)
proj['declaration_lag_days']   = proj['declaration_lag_days'].clip(lower=0)
disas['declaration_lag_days']  = disas['declaration_lag_days'].clip(lower=0)
proj['incident_duration_days'] = proj['incident_duration_days'].clip(lower=0)
disas['incident_duration_days']= disas['incident_duration_days'].clip(lower=0)

print('Time features added.')
proj[['declaration_lag_days','incident_duration_days','incident_season','incident_year']].head(3)


Time features added.


,declaration_lag_days,incident_duration_days,incident_season,incident_year
0,4,9,Summer,1998
1,4,9,Summer,1998
2,4,9,Summer,1998


## 2.5 Feature Engineering — Prior Disaster Count
For each disaster, count how many PA disasters hit the same state in the preceding 5 years.
Computed at disaster level (efficient), then joined back to project level.


In [6]:
# Compute on disaster-level (faster — ~thousands of rows vs ~800k)
disas = add_prior_disasters(
    disas,
    state_col='stateAbbreviation',
    date_col='incidentBeginDate',
    window_years=5
)
print('Prior disaster counts computed.')
print(disas[['disasterNumber','stateAbbreviation','incidentBeginDate','prior_disasters_5yr']].head(5))

# Join back to project level on disasterNumber
proj = proj.merge(
    disas[['disasterNumber', 'prior_disasters_5yr']],
    on='disasterNumber', how='left'
)
proj['prior_disasters_5yr'] = proj['prior_disasters_5yr'].fillna(0).astype(int)
print('Joined prior_disasters_5yr to project level.')


Prior disaster counts computed.
   disasterNumber stateAbbreviation         incidentBeginDate  \
0            1239                TX 1998-08-22 00:00:00+00:00   
8            1268                WY 1998-10-05 00:00:00+00:00   
1            1257                TX 1998-10-17 00:00:00+00:00   
5            1264                LA 1998-12-22 00:00:00+00:00   
2            1260                TN 1998-12-23 00:00:00+00:00   

   prior_disasters_5yr  
0                    0  
8                    0  
1                    1  
5                    0  
2                    0  
Joined prior_disasters_5yr to project level.


## 2.5b CPI Inflation Adjustment
The funding tier thresholds ($10k / $131.1k / $1M for projects; $1M / $50M / $500M for disasters)
are anchored to **2019 dollars**. A project worth $100k in 1998 is worth ~$157k in 2019 dollars
and should be classified as Large, not Small.

We convert all dollar amounts to 2019 constant dollars before binning.
Source: BLS CPI-U Annual Averages (1982–84 = 100 base).

In [ ]:
# BLS CPI-U Annual Averages (1982-84=100). Base year = 2019 (thresholds anchored here)
CPI_BY_YEAR = {
    1998: 163.0, 1999: 166.6, 2000: 172.2, 2001: 177.1, 2002: 179.9,
    2003: 184.0, 2004: 188.9, 2005: 195.3, 2006: 201.6, 2007: 207.3,
    2008: 215.3, 2009: 214.5, 2010: 218.1, 2011: 224.9, 2012: 229.6,
    2013: 233.0, 2014: 236.7, 2015: 237.0, 2016: 240.0, 2017: 245.1,
    2018: 251.1, 2019: 255.7, 2020: 258.8, 2021: 271.0, 2022: 292.7,
    2023: 304.7, 2024: 314.2, 2025: 322.0, 2026: 329.0,
}
CPI_BASE = CPI_BY_YEAR[2019]  # 255.7

# Project-level: adjust federalShareObligated to 2019 dollars
proj['cpi_factor'] = proj['incident_year'].map(CPI_BY_YEAR).fillna(CPI_BASE)
proj['federalShareObligated_2019'] = proj['federalShareObligated'] * (CPI_BASE / proj['cpi_factor'])
proj.drop(columns=['cpi_factor'], inplace=True)

# Disaster-level: adjust total_federal_share to 2019 dollars
disas['cpi_factor'] = disas['incident_year'].map(CPI_BY_YEAR).fillna(CPI_BASE)
disas['total_federal_share_2019'] = disas['total_federal_share'] * (CPI_BASE / disas['cpi_factor'])
disas.drop(columns=['cpi_factor'], inplace=True)

print('CPI adjustment applied.')
print(f'  Project — raw median:      ${proj["federalShareObligated"].median():>12,.0f}')
print(f'  Project — adj median:      ${proj["federalShareObligated_2019"].median():>12,.0f}')
print(f'  Disaster — raw median:     ${disas["total_federal_share"].median():>12,.0f}')
print(f'  Disaster — adj median:     ${disas["total_federal_share_2019"].median():>12,.0f}')

## 2.6 Create Funding Tier Labels (Classification Target)

Instead of predicting a raw dollar amount, we classify each grant into one of 4 tiers
anchored to real US federal policy thresholds:

**Project-level tiers** (FEMA PA threshold + Federal Acquisition Regulation):
| Class | Label | Range |
|---|---|---|
| 0 | Micro | < $10,000 |
| 1 | Small | $10,000 – $131,100 |
| 2 | Large | $131,100 – $1,000,000 |
| 3 | Major | > $1,000,000 |

**Disaster-level tiers** (total payout per declared disaster):
| Class | Label | Range |
|---|---|---|
| 0 | Minor | < $1M |
| 1 | Moderate | $1M – $50M |
| 2 | Major | $50M – $500M |
| 3 | Catastrophic | > $500M |

In [ ]:
# ── Project-level funding tier (using CPI-adjusted 2019 dollars) ─────────────
proj = proj[proj['federalShareObligated_2019'] > 0].copy()
proj['funding_tier'] = pd.cut(
    proj['federalShareObligated_2019'],
    bins=PROJECT_BINS,
    labels=[0, 1, 2, 3],
    right=False
).astype(int)

print('Project-level tier distribution (CPI-adjusted):')
tier_labels = {0:'Micro (<$10k)', 1:'Small ($10k–$131k)',
               2:'Large ($131k–$1M)', 3:'Major (>$1M)'}
counts = proj['funding_tier'].value_counts().sort_index()
for t, n in counts.items():
    print(f'  Tier {t} {tier_labels[t]:<25} {n:>8,}  ({100*n/len(proj):.1f}%)')

# ── Disaster-level funding tier (using CPI-adjusted 2019 dollars) ─────────────
disas = disas[disas['total_federal_share_2019'] > 0].copy()
disas['funding_tier'] = pd.cut(
    disas['total_federal_share_2019'],
    bins=DISASTER_BINS,
    labels=[0, 1, 2, 3],
    right=False
).astype(int)

print('\nDisaster-level tier distribution (CPI-adjusted):')
dis_labels = {0:'Minor (<$1M)', 1:'Moderate ($1M–$50M)',
              2:'Major ($50M–$500M)', 3:'Catastrophic (>$500M)'}
counts_d = disas['funding_tier'].value_counts().sort_index()
for t, n in counts_d.items():
    print(f'  Tier {t} {dis_labels[t]:<30} {n:>6,}  ({100*n/len(disas):.1f}%)')

## 2.7 Final Check & Save


In [8]:
data_summary(proj,  'Cleaned Project-Level')
data_summary(disas, 'Cleaned Disaster-Level')

proj.to_csv(PROCESSED  + 'cleaned_project_level.csv',  index=False)
disas.to_csv(PROCESSED + 'cleaned_disaster_level.csv', index=False)

print('Saved:')
print(f'  cleaned_project_level.csv   -> {proj.shape}  | target: funding_tier (0-3)')
print(f'  cleaned_disaster_level.csv  -> {disas.shape} | target: funding_tier (0-3)')


  Cleaned Project-Level  |  809,734 rows  x  33 cols
Columns with nulls:
  applicantId                                5  (0.0%)
  county                                13,614  (1.7%)
  risk_rating                           25,992  (3.2%)


  Cleaned Disaster-Level  |  1,766 rows  x  20 cols
  No nulls  ✓

Saved:
  cleaned_project_level.csv   -> (809734, 33)  | target: funding_tier (0-3)
  cleaned_disaster_level.csv  -> (1766, 20) | target: funding_tier (0-3)
